In [0]:
%sql
-- Create gold schema
CREATE SCHEMA IF NOT EXISTS caracare_casestudy.`02_gold`
COMMENT 'Gold layer - business-level aggregates and analytics-ready tables';

In [0]:
%sql
-- Gold Table 1: Patient Prescription Summary (Denormalized)
-- Wide table joining patient, prescription, doctor, and insurance for reporting
CREATE OR REPLACE TABLE caracare_casestudy.`02_gold`.patient_prescription_summary
USING DELTA
COMMENT 'Denormalized patient prescription data with all related dimensions'
AS
SELECT 
    -- Patient info
    p.patient_id,
    p.patient_email,
    p.date_of_birth,
    YEAR(CURRENT_DATE()) - YEAR(p.date_of_birth) AS patient_age,
    
    -- Prescription info
    fp.prescription_id,
    fp.prescription_start,
    fp.prescription_end,
    fp.prescription_status,
    fp.represcription,
    fp.represcription_date,
    DATEDIFF(fp.prescription_end, fp.prescription_start) AS prescription_duration_days,
    
    -- Doctor info
    d.doctor_id,
    d.prescribing_doctor,
    d.doctor_specialty,
    d.doctor_city,
    
    -- Insurance info
    i.insurance_id,
    i.insurance_name,
    i.insurance_type
    
FROM caracare_casestudy.`01_silver`.fact_prescription fp
INNER JOIN caracare_casestudy.`01_silver`.dim_patient p 
    ON fp.patient_id = p.patient_id
INNER JOIN caracare_casestudy.`01_silver`.dim_doctor d 
    ON fp.doctor_id = d.doctor_id
INNER JOIN caracare_casestudy.`01_silver`.dim_insurance i 
    ON fp.insurance_id = i.insurance_id;

In [0]:
%sql
-- Gold Table 2: Doctor Performance Metrics
-- Aggregated statistics per doctor
CREATE OR REPLACE TABLE caracare_casestudy.`02_gold`.doctor_performance
USING DELTA
COMMENT 'Aggregated doctor performance metrics'
AS
SELECT 
    d.doctor_id,
    d.prescribing_doctor,
    d.doctor_specialty,
    d.doctor_city,
    
    -- Prescription metrics
    COUNT(DISTINCT fp.prescription_id) AS total_prescriptions,
    COUNT(DISTINCT fp.patient_id) AS unique_patients,
    SUM(CASE WHEN fp.represcription = TRUE THEN 1 ELSE 0 END) AS represcription_count,
    ROUND(SUM(CASE WHEN fp.represcription = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS represcription_rate_pct,
    
    -- Status breakdown
    SUM(CASE WHEN fp.prescription_status = 'ACTIVE' THEN 1 ELSE 0 END) AS active_prescriptions,
    SUM(CASE WHEN fp.prescription_status = 'COMPLETED' THEN 1 ELSE 0 END) AS completed_prescriptions,
    SUM(CASE WHEN fp.prescription_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_prescriptions,
    
    -- Duration metrics
    ROUND(AVG(DATEDIFF(fp.prescription_end, fp.prescription_start)), 2) AS avg_prescription_duration_days,
    MIN(fp.prescription_start) AS first_prescription_date,
    MAX(fp.prescription_start) AS latest_prescription_date
    
FROM caracare_casestudy.`01_silver`.dim_doctor d
LEFT JOIN caracare_casestudy.`01_silver`.fact_prescription fp 
    ON d.doctor_id = fp.doctor_id
GROUP BY 
    d.doctor_id,
    d.prescribing_doctor,
    d.doctor_specialty,
    d.doctor_city;

In [0]:
%sql
-- Gold Table 3: Insurance Utilization Metrics
-- Aggregated statistics per insurance provider
CREATE OR REPLACE TABLE caracare_casestudy.`02_gold`.insurance_utilization
USING DELTA
COMMENT 'Aggregated insurance utilization metrics'
AS
SELECT 
    i.insurance_id,
    i.insurance_name,
    i.insurance_type,
    
    -- Prescription metrics
    COUNT(DISTINCT fp.prescription_id) AS total_prescriptions,
    COUNT(DISTINCT fp.patient_id) AS unique_patients,
    COUNT(DISTINCT fp.doctor_id) AS unique_doctors,
    
    -- Represcription analysis
    SUM(CASE WHEN fp.represcription = TRUE THEN 1 ELSE 0 END) AS represcription_count,
    ROUND(SUM(CASE WHEN fp.represcription = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS represcription_rate_pct,
    
    -- Status breakdown
    SUM(CASE WHEN fp.prescription_status = 'ACTIVE' THEN 1 ELSE 0 END) AS active_prescriptions,
    SUM(CASE WHEN fp.prescription_status = 'COMPLETED' THEN 1 ELSE 0 END) AS completed_prescriptions,
    SUM(CASE WHEN fp.prescription_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_prescriptions,
    ROUND(SUM(CASE WHEN fp.prescription_status = 'CANCELLED' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct,
    
    -- Duration metrics
    ROUND(AVG(DATEDIFF(fp.prescription_end, fp.prescription_start)), 2) AS avg_prescription_duration_days
    
FROM caracare_casestudy.`01_silver`.dim_insurance i
LEFT JOIN caracare_casestudy.`01_silver`.fact_prescription fp 
    ON i.insurance_id = fp.insurance_id
GROUP BY 
    i.insurance_id,
    i.insurance_name,
    i.insurance_type;

In [0]:
%sql
-- Gold Table 4: Patient Summary Metrics
-- Aggregated statistics per patient
CREATE OR REPLACE TABLE caracare_casestudy.`02_gold`.patient_summary
USING DELTA
COMMENT 'Aggregated patient summary metrics'
AS
SELECT 
    p.patient_id,
    p.patient_email,
    p.date_of_birth,
    YEAR(CURRENT_DATE()) - YEAR(p.date_of_birth) AS patient_age,
    
    -- Prescription metrics
    COUNT(DISTINCT fp.prescription_id) AS total_prescriptions,
    COUNT(DISTINCT fp.doctor_id) AS unique_doctors,
    COUNT(DISTINCT fp.insurance_id) AS unique_insurance_providers,
    
    -- Represcription analysis
    SUM(CASE WHEN fp.represcription = TRUE THEN 1 ELSE 0 END) AS represcription_count,
    ROUND(SUM(CASE WHEN fp.represcription = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS represcription_rate_pct,
    
    -- Prescription status
    SUM(CASE WHEN fp.prescription_status = 'ACTIVE' THEN 1 ELSE 0 END) AS active_prescriptions,
    SUM(CASE WHEN fp.prescription_status = 'COMPLETED' THEN 1 ELSE 0 END) AS completed_prescriptions,
    
    -- Duration metrics
    ROUND(AVG(DATEDIFF(fp.prescription_end, fp.prescription_start)), 2) AS avg_prescription_duration_days,
    MIN(fp.prescription_start) AS first_prescription_date,
    MAX(fp.prescription_start) AS latest_prescription_date,
    DATEDIFF(MAX(fp.prescription_start), MIN(fp.prescription_start)) AS patient_history_days
    
FROM caracare_casestudy.`01_silver`.dim_patient p
LEFT JOIN caracare_casestudy.`01_silver`.fact_prescription fp 
    ON p.patient_id = fp.patient_id
GROUP BY 
    p.patient_id,
    p.patient_email,
    p.date_of_birth;

In [0]:
%sql
-- Gold Table 5: Touchpoint Effectiveness Analysis
-- Aggregated touchpoint metrics by type and channel
CREATE OR REPLACE TABLE caracare_casestudy.`02_gold`.touchpoint_effectiveness
USING DELTA
COMMENT 'Aggregated touchpoint effectiveness metrics by type and channel'
AS
SELECT 
    ft.touchpoint_type,
    ft.touchpoint_channel,
    
    -- Volume metrics
    COUNT(DISTINCT ft.touchpoint_id) AS total_touchpoints,
    COUNT(DISTINCT ft.prescription_id) AS unique_prescriptions,
    COUNT(DISTINCT fp.patient_id) AS unique_patients,
    
    -- Outcome analysis
    SUM(CASE WHEN ft.touchpoint_outcome = 'Success' THEN 1 ELSE 0 END) AS success_count,
    SUM(CASE WHEN ft.touchpoint_outcome = 'Failed' THEN 1 ELSE 0 END) AS failed_count,
    SUM(CASE WHEN ft.touchpoint_outcome = 'No Response' THEN 1 ELSE 0 END) AS no_response_count,
    ROUND(SUM(CASE WHEN ft.touchpoint_outcome = 'Success' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS success_rate_pct,
    
    -- Timing metrics
    MIN(ft.touchpoint_date) AS first_touchpoint_date,
    MAX(ft.touchpoint_date) AS latest_touchpoint_date,
    ROUND(AVG(DATEDIFF(ft.touchpoint_date, fp.prescription_start)), 2) AS avg_days_from_prescription_start
    
FROM caracare_casestudy.`01_silver`.fact_touchpoint ft
INNER JOIN caracare_casestudy.`01_silver`.fact_prescription fp 
    ON ft.prescription_id = fp.prescription_id
GROUP BY 
    ft.touchpoint_type,
    ft.touchpoint_channel;

In [0]:
%sql
-- Gold Table 6: Prescription Touchpoint Details
-- Detailed view of prescriptions with their associated touchpoints
CREATE OR REPLACE TABLE caracare_casestudy.`02_gold`.prescription_touchpoint_details
USING DELTA
COMMENT 'Detailed prescription and touchpoint data for analysis'
AS
SELECT 
    -- Prescription info
    fp.prescription_id,
    fp.patient_id,
    fp.prescription_start,
    fp.prescription_end,
    fp.prescription_status,
    fp.represcription,
    
    -- Touchpoint info
    ft.touchpoint_id,
    ft.touchpoint_date,
    ft.touchpoint_type,
    ft.touchpoint_channel,
    ft.touchpoint_outcome,
    
    -- Calculated metrics
    DATEDIFF(ft.touchpoint_date, fp.prescription_start) AS days_from_prescription_start,
    CASE 
        WHEN DATEDIFF(ft.touchpoint_date, fp.prescription_start) <= 7 THEN 'Week 1'
        WHEN DATEDIFF(ft.touchpoint_date, fp.prescription_start) <= 14 THEN 'Week 2'
        WHEN DATEDIFF(ft.touchpoint_date, fp.prescription_start) <= 30 THEN 'Month 1'
        ELSE 'After Month 1'
    END AS touchpoint_timing_bucket,
    
    -- Doctor and insurance context
    d.prescribing_doctor,
    d.doctor_specialty,
    i.insurance_name,
    i.insurance_type
    
FROM caracare_casestudy.`01_silver`.fact_prescription fp
INNER JOIN caracare_casestudy.`01_silver`.fact_touchpoint ft 
    ON fp.prescription_id = ft.prescription_id
INNER JOIN caracare_casestudy.`01_silver`.dim_doctor d 
    ON fp.doctor_id = d.doctor_id
INNER JOIN caracare_casestudy.`01_silver`.dim_insurance i 
    ON fp.insurance_id = i.insurance_id;

In [0]:
%sql
-- Validation: Check row counts for all gold tables
SELECT 
    'patient_prescription_summary' AS table_name,
    COUNT(*) AS row_count
FROM caracare_casestudy.`02_gold`.patient_prescription_summary

UNION ALL

SELECT 
    'doctor_performance' AS table_name,
    COUNT(*) AS row_count
FROM caracare_casestudy.`02_gold`.doctor_performance

UNION ALL

SELECT 
    'insurance_utilization' AS table_name,
    COUNT(*) AS row_count
FROM caracare_casestudy.`02_gold`.insurance_utilization

UNION ALL

SELECT 
    'patient_summary' AS table_name,
    COUNT(*) AS row_count
FROM caracare_casestudy.`02_gold`.patient_summary

UNION ALL

SELECT 
    'touchpoint_effectiveness' AS table_name,
    COUNT(*) AS row_count
FROM caracare_casestudy.`02_gold`.touchpoint_effectiveness

UNION ALL

SELECT 
    'prescription_touchpoint_details' AS table_name,
    COUNT(*) AS row_count
FROM caracare_casestudy.`02_gold`.prescription_touchpoint_details

ORDER BY table_name;

In [0]:
%sql
/*
=============================================================================
TASK 2: MONTHLY REPRESCRIPTION RATE
=============================================================================
Calculates the monthly re-prescription rate based on the object model.

Metric Definition:
- Represcription Rate = (Number of Represcriptions / Total Prescriptions) * 100
- Aggregated by month and year

Note: This uses Databricks SQL syntax. BigQuery equivalent would use:
- FORMAT_DATE('%Y-%m', date_column) instead of DATE_FORMAT
- DATE_TRUNC(date_column, MONTH) for month grouping
*/

CREATE OR REPLACE TABLE caracare_casestudy.`02_gold`.monthly_represcription_rate
USING DELTA
COMMENT 'Task 2: Monthly represcription rate analysis'
AS
SELECT 
    -- Time dimensions
    YEAR(prescription_start) AS prescription_year,
    MONTH(prescription_start) AS prescription_month,
    DATE_FORMAT(prescription_start, 'yyyy-MM') AS year_month,
    DATE_TRUNC('MONTH', prescription_start) AS month_start_date,
    
    -- Volume metrics
    COUNT(DISTINCT prescription_id) AS total_prescriptions,
    SUM(CASE WHEN represcription = TRUE THEN 1 ELSE 0 END) AS represcription_count,
    SUM(CASE WHEN represcription = FALSE OR represcription IS NULL THEN 1 ELSE 0 END) AS initial_prescription_count,
    
    -- Rate calculation
    ROUND(
        SUM(CASE WHEN represcription = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(DISTINCT prescription_id),
        2
    ) AS represcription_rate_pct,
    
    -- Unique entities
    COUNT(DISTINCT patient_id) AS unique_patients,
    COUNT(DISTINCT doctor_id) AS unique_doctors,
    COUNT(DISTINCT insurance_id) AS unique_insurance_providers
    
FROM caracare_casestudy.`01_silver`.fact_prescription
GROUP BY 
    YEAR(prescription_start),
    MONTH(prescription_start),
    DATE_FORMAT(prescription_start, 'yyyy-MM'),
    DATE_TRUNC('MONTH', prescription_start)
ORDER BY 
    prescription_year,
    prescription_month;

In [0]:
%sql
-- Task 2: Query Results - Monthly Represcription Rate
SELECT 
    year_month,
    month_start_date,
    total_prescriptions,
    represcription_count,
    initial_prescription_count,
    represcription_rate_pct,
    unique_patients,
    unique_doctors
FROM caracare_casestudy.`02_gold`.monthly_represcription_rate
ORDER BY month_start_date;

# Task 3: Touchpoint Attribution to Represcription

## Business Question
**"Which touchpoint contributes the most to re-prescription?"**

## Analytical Approach

We will use multiple attribution models to understand touchpoint effectiveness:

### 1. **Last-Touch Attribution**
- Assigns 100% credit to the last touchpoint before represcription
- Simple to implement and understand
- Assumption: The final interaction is most influential

### 2. **First-Touch Attribution**  
- Assigns 100% credit to the first touchpoint in the prescription journey
- Identifies initial engagement drivers
- Assumption: First interaction sets the stage for represcription

### 3. **Multi-Touch Analysis**
- Examines all touchpoints associated with represcription cases
- Provides frequency and success rate by touchpoint type/channel
- No single attribution weight, but shows overall patterns

### 4. **Conversion Rate by Touchpoint**
- Compare represcription rate for prescriptions WITH specific touchpoints vs WITHOUT
- Shows incremental impact of each touchpoint type

---

## Key Limitations

### 1. **Causation vs Correlation**
- We can only observe correlation between touchpoints and represcription
- Cannot prove causation without randomized control trials
- Touchpoints may be correlated with other factors (e.g., patient health status)

### 2. **Selection Bias**
- Patients receiving certain touchpoints may already be more likely to get represcriptions
- E.g., phone calls may target high-risk patients who need more support
- Not all patients receive the same touchpoint mix

### 3. **Temporal Ordering**
- Touchpoints occur AFTER prescription start, but represcription decision timing is unclear
- We don't know when the represcription decision was actually made
- Late touchpoints may occur after decision is already made

### 4. **Missing Variables**
- Patient health outcomes not captured
- Medication type/severity not available  
- External factors (insurance changes, side effects) not tracked
- Doctor recommendation strength not measured

### 5. **Attribution Model Limitations**
- Last-touch ignores earlier influential touchpoints
- First-touch ignores follow-up nurturing
- Multi-touch doesn't weight relative importance
- Simple presence/absence doesn't capture touchpoint quality or content

### 6. **Data Completeness**
- Not all touchpoints may be logged (offline conversations, informal check-ins)
- Touchpoint outcome quality varies ("Success" may mean different things)
- Missing data on touchpoint content and personalization

---

## Recommendation

Use **multiple attribution models** together to triangulate insights:
1. Last-touch for operational optimization (what seals the deal?)
2. Multi-touch frequency for resource allocation (what channels to invest in?)
3. Conversion rate lift for strategic prioritization (what has measurable impact?)

Then validate findings with A/B tests or quasi-experimental designs.

In [0]:
%sql
/*
=============================================================================
TASK 3: TOUCHPOINT ATTRIBUTION ANALYSIS
=============================================================================
Model 1: Last-Touch Attribution
Assigns credit to the last touchpoint before represcription date
*/

CREATE OR REPLACE TABLE caracare_casestudy.`02_gold`.touchpoint_attribution_last_touch
USING DELTA
COMMENT 'Task 3: Last-touch attribution for represcription'
AS
WITH represcription_cases AS (
    -- Get only prescriptions that have represcription = TRUE
    SELECT 
        prescription_id,
        patient_id,
        prescription_start,
        represcription_date
    FROM caracare_casestudy.`01_silver`.fact_prescription
    WHERE represcription = TRUE
      AND represcription_date IS NOT NULL
),
last_touchpoint AS (
    -- Get the last touchpoint before represcription date for each prescription
    SELECT 
        ft.prescription_id,
        ft.touchpoint_type,
        ft.touchpoint_channel,
        ft.touchpoint_outcome,
        ft.touchpoint_date,
        rc.represcription_date,
        ROW_NUMBER() OVER (
            PARTITION BY ft.prescription_id 
            ORDER BY ft.touchpoint_date DESC
        ) AS touchpoint_rank
    FROM caracare_casestudy.`01_silver`.fact_touchpoint ft
    INNER JOIN represcription_cases rc 
        ON ft.prescription_id = rc.prescription_id
    WHERE ft.touchpoint_date <= rc.represcription_date
)
SELECT 
    touchpoint_type,
    touchpoint_channel,
    touchpoint_outcome,
    COUNT(DISTINCT prescription_id) AS attributed_represcriptions,
    ROUND(COUNT(DISTINCT prescription_id) * 100.0 / SUM(COUNT(DISTINCT prescription_id)) OVER (), 2) AS attribution_pct,
    ROUND(AVG(DATEDIFF(represcription_date, touchpoint_date)), 1) AS avg_days_before_represcription
FROM last_touchpoint
WHERE touchpoint_rank = 1  -- Only the last touchpoint
GROUP BY 
    touchpoint_type,
    touchpoint_channel,
    touchpoint_outcome
ORDER BY 
    attributed_represcriptions DESC;

In [0]:
%sql
/*
=============================================================================
TASK 3: TOUCHPOINT ATTRIBUTION ANALYSIS
=============================================================================
Model 2: First-Touch Attribution
Assigns credit to the first touchpoint after prescription start
*/

CREATE OR REPLACE TABLE caracare_casestudy.`02_gold`.touchpoint_attribution_first_touch
USING DELTA
COMMENT 'Task 3: First-touch attribution for represcription'
AS
WITH represcription_cases AS (
    -- Get only prescriptions that have represcription = TRUE
    SELECT 
        prescription_id,
        patient_id,
        prescription_start
    FROM caracare_casestudy.`01_silver`.fact_prescription
    WHERE represcription = TRUE
),
first_touchpoint AS (
    -- Get the first touchpoint after prescription start for each prescription
    SELECT 
        ft.prescription_id,
        ft.touchpoint_type,
        ft.touchpoint_channel,
        ft.touchpoint_outcome,
        ft.touchpoint_date,
        rc.prescription_start,
        ROW_NUMBER() OVER (
            PARTITION BY ft.prescription_id 
            ORDER BY ft.touchpoint_date ASC
        ) AS touchpoint_rank
    FROM caracare_casestudy.`01_silver`.fact_touchpoint ft
    INNER JOIN represcription_cases rc 
        ON ft.prescription_id = rc.prescription_id
    WHERE ft.touchpoint_date >= rc.prescription_start
)
SELECT 
    touchpoint_type,
    touchpoint_channel,
    touchpoint_outcome,
    COUNT(DISTINCT prescription_id) AS attributed_represcriptions,
    ROUND(COUNT(DISTINCT prescription_id) * 100.0 / SUM(COUNT(DISTINCT prescription_id)) OVER (), 2) AS attribution_pct,
    ROUND(AVG(DATEDIFF(touchpoint_date, prescription_start)), 1) AS avg_days_after_prescription_start
FROM first_touchpoint
WHERE touchpoint_rank = 1  -- Only the first touchpoint
GROUP BY 
    touchpoint_type,
    touchpoint_channel,
    touchpoint_outcome
ORDER BY 
    attributed_represcriptions DESC;

In [0]:
%sql
/*
=============================================================================
TASK 3: TOUCHPOINT ATTRIBUTION ANALYSIS
=============================================================================
Model 3: Multi-Touch Frequency Analysis
Shows all touchpoints associated with represcription vs non-represcription
*/

CREATE OR REPLACE TABLE caracare_casestudy.`02_gold`.touchpoint_attribution_multitouch
USING DELTA
COMMENT 'Task 3: Multi-touch frequency analysis for represcription'
AS
SELECT 
    ft.touchpoint_type,
    ft.touchpoint_channel,
    ft.touchpoint_outcome,
    
    -- Represcription metrics
    COUNT(DISTINCT CASE WHEN fp.represcription = TRUE THEN ft.touchpoint_id END) AS touchpoints_with_represcription,
    COUNT(DISTINCT CASE WHEN fp.represcription = FALSE OR fp.represcription IS NULL THEN ft.touchpoint_id END) AS touchpoints_without_represcription,
    COUNT(DISTINCT ft.touchpoint_id) AS total_touchpoints,
    
    -- Prescription-level metrics
    COUNT(DISTINCT CASE WHEN fp.represcription = TRUE THEN ft.prescription_id END) AS prescriptions_with_represcription,
    COUNT(DISTINCT CASE WHEN fp.represcription = FALSE OR fp.represcription IS NULL THEN ft.prescription_id END) AS prescriptions_without_represcription,
    
    -- Represcription rate when this touchpoint is present
    ROUND(
        COUNT(DISTINCT CASE WHEN fp.represcription = TRUE THEN ft.prescription_id END) * 100.0 /
        COUNT(DISTINCT ft.prescription_id),
        2
    ) AS represcription_rate_with_touchpoint_pct,
    
    -- Success rate of this touchpoint
    ROUND(
        COUNT(DISTINCT CASE WHEN ft.touchpoint_outcome = 'Success' THEN ft.touchpoint_id END) * 100.0 /
        COUNT(DISTINCT ft.touchpoint_id),
        2
    ) AS touchpoint_success_rate_pct,
    
    -- Average touchpoints per prescription
    ROUND(COUNT(ft.touchpoint_id) * 1.0 / COUNT(DISTINCT ft.prescription_id), 2) AS avg_touchpoints_per_prescription
    
FROM caracare_casestudy.`01_silver`.fact_touchpoint ft
INNER JOIN caracare_casestudy.`01_silver`.fact_prescription fp
    ON ft.prescription_id = fp.prescription_id
GROUP BY 
    ft.touchpoint_type,
    ft.touchpoint_channel,
    ft.touchpoint_outcome
ORDER BY 
    represcription_rate_with_touchpoint_pct DESC;

In [0]:
%sql
/*
=============================================================================
TASK 3: TOUCHPOINT ATTRIBUTION ANALYSIS
=============================================================================
Model 4: Conversion Rate Lift Analysis
Compares represcription rates WITH vs WITHOUT each touchpoint type/channel
Shows incremental impact of each touchpoint
*/

CREATE OR REPLACE TABLE caracare_casestudy.`02_gold`.touchpoint_conversion_lift
USING DELTA
COMMENT 'Task 3: Conversion rate lift analysis for touchpoint types'
AS
WITH touchpoint_presence AS (
    -- Flag which touchpoint types each prescription received
    SELECT DISTINCT
        fp.prescription_id,
        fp.represcription,
        ft.touchpoint_type,
        ft.touchpoint_channel
    FROM caracare_casestudy.`01_silver`.fact_prescription fp
    LEFT JOIN caracare_casestudy.`01_silver`.fact_touchpoint ft
        ON fp.prescription_id = ft.prescription_id
),
baseline AS (
    -- Overall represcription rate (no touchpoint filter)
    SELECT 
        COUNT(DISTINCT CASE WHEN represcription = TRUE THEN prescription_id END) * 100.0 /
        COUNT(DISTINCT prescription_id) AS baseline_represcription_rate_pct
    FROM caracare_casestudy.`01_silver`.fact_prescription
)
SELECT 
    touchpoint_type,
    touchpoint_channel,
    
    -- Prescriptions WITH this touchpoint
    COUNT(DISTINCT prescription_id) AS prescriptions_with_touchpoint,
    COUNT(DISTINCT CASE WHEN represcription = TRUE THEN prescription_id END) AS represcriptions_with_touchpoint,
    ROUND(
        COUNT(DISTINCT CASE WHEN represcription = TRUE THEN prescription_id END) * 100.0 /
        COUNT(DISTINCT prescription_id),
        2
    ) AS represcription_rate_with_touchpoint_pct,
    
    -- Baseline comparison
    (SELECT baseline_represcription_rate_pct FROM baseline) AS baseline_represcription_rate_pct,
    
    -- Lift calculation
    ROUND(
        (
            COUNT(DISTINCT CASE WHEN represcription = TRUE THEN prescription_id END) * 100.0 /
            COUNT(DISTINCT prescription_id)
        ) - (SELECT baseline_represcription_rate_pct FROM baseline),
        2
    ) AS lift_vs_baseline_pct_points,
    
    -- Relative lift percentage
    ROUND(
        (
            (
                COUNT(DISTINCT CASE WHEN represcription = TRUE THEN prescription_id END) * 100.0 /
                COUNT(DISTINCT prescription_id)
            ) - (SELECT baseline_represcription_rate_pct FROM baseline)
        ) * 100.0 / (SELECT baseline_represcription_rate_pct FROM baseline),
        2
    ) AS relative_lift_pct
    
FROM touchpoint_presence
WHERE touchpoint_type IS NOT NULL  -- Only prescriptions that have touchpoints
GROUP BY 
    touchpoint_type,
    touchpoint_channel
ORDER BY 
    lift_vs_baseline_pct_points DESC;

In [0]:
%sql
-- Summary: Task 2 & Task 3 Validation
-- Check row counts for all attribution models

SELECT 'Task 2: Monthly Represcription Rate' AS analysis, COUNT(*) AS row_count
FROM caracare_casestudy.`02_gold`.monthly_represcription_rate

UNION ALL

SELECT 'Task 3: Last-Touch Attribution' AS analysis, COUNT(*) AS row_count
FROM caracare_casestudy.`02_gold`.touchpoint_attribution_last_touch

UNION ALL

SELECT 'Task 3: First-Touch Attribution' AS analysis, COUNT(*) AS row_count
FROM caracare_casestudy.`02_gold`.touchpoint_attribution_first_touch

UNION ALL

SELECT 'Task 3: Multi-Touch Analysis' AS analysis, COUNT(*) AS row_count
FROM caracare_casestudy.`02_gold`.touchpoint_attribution_multitouch

UNION ALL

SELECT 'Task 3: Conversion Rate Lift' AS analysis, COUNT(*) AS row_count
FROM caracare_casestudy.`02_gold`.touchpoint_conversion_lift

ORDER BY analysis;